<h1>DOCUMET LOADER</H1>

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
import os


def document_loader(folder_path):

    document = [] 

    for filename in os.listdir(folder_path):

        if filename.endswith('.docx'):
            file_path = os.path.join(folder_path,filename)

            loader = Docx2txtLoader(file_path)
            pages = loader.load()

            document.extend(pages)

    return document

<H1>TEXT SPLITTER</H1>

In [113]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_splitter(document):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 50
    )

    chunk = text_splitter.split_documents(document)

    return chunk

<h1>EMBEDDING AND VECTOR DATA BASE</H1|>

In [114]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

def create_vector_db(chunk):   

    embedding = HuggingFaceEmbeddings(
        model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    )

    vector_db = FAISS.from_documents(   # semantic  search
        chunk,
        embedding
    )

    return vector_db

<h1>KEYWORD SEARCH</h1>

In [115]:
from rank_bm25 import BM25Okapi

class BM25ORetriver:
    def __init__(self,chunk):
        self.chunk = chunk 

        tokenized_chunk = []

        for c in chunk:
            tokens = c.page_content.lower().split()
            tokenized_chunk.append(tokens)


        self.bm25 = BM25Okapi(tokenized_chunk)

    def search(self,query,k=5):
        token_query = query.lower().split()
        scores = self.bm25.get_scores(token_query)

        ranked_answer = sorted(range(len(scores)),
                                key = lambda index: scores[index],
                                reverse=True)

        top_answers = ranked_answer[:k]

        result = []
        
        for c in top_answers:
            result.append(self.chunk[c])


        return result


<h2>hybrid_retriever</h2>

In [116]:
class HybridSearch:

    def __init__(self,vector_db,bm25):
        self.vector_db = vector_db
        self.bm25 = bm25


    def search(self,query,k=5):
        vector_result = self.vector_db.similarity_search(query,k=k)
        bm25_result = self.bm25.search(query,k=k)

        combined_result = vector_result+bm25_result


        unique_document = []
        seen_content = set()


        for document in combined_result:
            if document.page_content not in combined_result:
                unique_document.append(document)

                seen_content.add(document.page_content)

        return unique_document

<h1>RERANKER</h1>

In [117]:
from sentence_transformers import CrossEncoder

class RERANKER:
    def __init__(self):
        self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rerank(self,query,documents,top_k=5):

        pairs = []

        for c in documents:
            pairs.append(
                [
                    query,
                    c.page_content
                ]
            )
        scores = self.model.predict(pairs)


        ranked_answers = sorted(zip(scores,documents),
                                key = lambda i:i[0],
                                reverse=True)

        top_answer = ranked_answers[:top_k]


        results = []

        for scores,document in top_answer:
            results.append(document)

        return results

<h1>PROMPT</h1>

In [118]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv

def generate_answers(query,document):


        context = ''
        for c in document:
            context+= c.page_content
            context += '\n\n'

        prompt = ChatPromptTemplate.from_template(
              '''
              
              You are a research assistant.

              only answer from the context.

              if the question is out of context. just say,
              "Question out of context"


              context
              {context}

              question
              {question}
              '''
        )

        llm_model = ChatGroq(
              model = 'openai/gpt-oss-20b',
              temperature=0
        )

        chain = prompt | llm_model


        reponse = chain.invoke(
              {
                    'context': context,
                    'question':query
              }
        )


        return reponse.content


<H1>APP</H1>

In [121]:
import gradio as gr


folder_path = 'E:\GEN-AI-PROJECTS'

# load the data
document =  document_loader(folder_path)

# chunk the data
chunk = text_splitter(document)

# embedding 
vector_db = create_vector_db(chunk)

# bm25 retriver
bm_25O_retriver = BM25ORetriver(chunk)

#hybrid search
hybrid_search = HybridSearch(vector_db,bm_25O_retriver)

#ranker
reranker = RERANKER()

def answer_question(query):
    retrived_documents = hybrid_search.search(
        query,
        k=10
    )


    rerenaker_final  = reranker.rerank(
        query,
        retrived_documents,
        top_k=3
        
    )

    answer = generate_answers(query,rerenaker_final)


    sources = ""
    for i,document in enumerate(
        retrived_documents
    ):

        page = document.metadata.get(
            'page',
            'unknown'
        )

        sources += f'\nSource {i+1}: Page {page}'

        final_response = answer
        final_response+='\n\nsources:'
        final_response+=sources

        return final_response

demo = gr.Interface(
        fn = answer_question,
        inputs = gr.Textbox(
            label = 'Ask user a question'
        ),
        outputs = gr.Textbox(
            label = 'Answer',
            lines = 15
        ),
        title = 'DEEP KNOWLWDGE',
        description='Advanced RAG Knowledge Base'
    )

demo.launch(share = True)

    


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7868

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
